# Understanding Vision-Language Model Inference: A Deep Dive with Qwen3-VL-2B

This notebook is a **learn-by-doing** tutorial that takes you inside a Vision-Language Model (VLM).
We will use **Qwen3-VL-2B-Instruct** -- the smallest model in the Qwen3-VL family -- to explore
every stage of VLM inference, from raw pixels to generated text.

### What You Will Learn

| Part | Topic |
|------|-------|
| **I** | VLM Architecture Overview -- how vision and language are fused |
| **II** | The Vision Encoder -- from pixels to patch embeddings |
| **III** | Vision-Language Fusion -- DeepStack, spatial merge, M-RoPE |
| **IV** | The Language Backbone -- Qwen3 decoder with visual context |
| **V** | End-to-End Inference -- running the full pipeline step by step |
| **VI** | Comparing with text-only Qwen3-0.6B (nano-vllm) |

### Prerequisites

- A CUDA-capable GPU (8GB+ VRAM recommended)
- Basic understanding of Transformers (attention, FFN, embeddings)
- Familiarity with PyTorch

---
## 0. Setup

Install the required packages. Qwen3-VL requires `transformers >= 4.57.0` (install from source if not yet released).

In [ ]:
# Install dependencies (run once)
# !pip install git+https://github.com/huggingface/transformers
# !pip install torch torchvision accelerate qwen-vl-utils pillow

In [ ]:
import torch
import torch.nn.functional as F
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from PIL import Image
import requests
from io import BytesIO
import math
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 1. Load the Model and Processor

In [ ]:
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

# Load model in bfloat16
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

# The processor handles both text tokenization and image preprocessing
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"Model loaded: {MODEL_ID}")
print(f"Model dtype: {model.dtype}")
print(f"Device: {model.device}")

---
# Part I -- VLM Architecture Overview

A Vision-Language Model combines a **Vision Encoder** (processes images) with a **Language Model** (generates text). Here is the complete architecture of Qwen3-VL-2B:

```
                          Qwen3-VL-2B Architecture
                          ========================

   IMAGE INPUT                                      TEXT INPUT
   (H x W x 3)                                   (token_ids)
       |                                               |
       v                                               v
  +-----------+                                  +------------+
  | 3D Conv   |  patch_size=16                   | Embedding  |  vocab=151,936
  | Patchify  |  temporal_patch=2                | Lookup     |  dim=2048
  +-----------+                                  +------------+
       |                                               |
       v                                               |
  +-----------+                                        |
  | 24-layer  |  hidden=1024                           |
  | Vision    |  heads=16                              |
  | Encoder   |  GELU activation                       |
  | (ViT)     |  2D RoPE                               |
  +-----------+                                        |
   |   |   |   |                                       |
   |  L5  L11 L17  <-- DeepStack features              |
   |   |   |   |                                       |
   v   v   v   v                                       |
  +-----------+                                        |
  | Spatial   |  2x2 merge                             |
  | Merge     |  1024 -> 2048                          |
  +-----------+                                        |
       |                                               |
       +-----------------> SCATTER <-------------------+
                         (replace image_token                
                          placeholders with                  
                          visual embeddings)                 
                              |                              
                              v                              
                     +------------------+                    
                     |   28-layer LLM   |  hidden=2048       
                     |   Decoder with   |  heads=16 (8 KV)  
                     |   DeepStack      |  SwiGLU + RMSNorm 
                     |   Injection at   |  M-RoPE (3D pos)  
                     |   layers 5,11,17 |  262K context      
                     +------------------+                    
                              |                              
                              v                              
                     +------------------+                    
                     |    LM Head       |  -> 151,936 logits 
                     +------------------+                    
                              |                              
                              v                              
                         Next Token                          
```

### Key Parameters

| Component | Parameter | Value |
|-----------|-----------|-------|
| **Vision Encoder** | Depth | 24 layers |
| | Hidden size | 1024 |
| | Attention heads | 16 |
| | Patch size | 16x16 pixels |
| | Activation | GELU (tanh approx) |
| | Spatial merge | 2x2 (4 patches -> 1 token) |
| | DeepStack layers | 5, 11, 17 |
| **Language Model** | Layers | 28 |
| | Hidden size | 2048 |
| | Q heads / KV heads | 16 / 8 (GQA) |
| | Head dim | 128 |
| | FFN intermediate | 6144 |
| | Activation | SiLU (SwiGLU) |
| | Norm | RMSNorm (eps=1e-6) |
| | Context length | 262,144 tokens |
| | RoPE theta | 5,000,000 |
| | M-RoPE sections | [24, 20, 20] (temporal, height, width) |
| **Vocabulary** | Size | 151,936 |
| | Tied embeddings | Yes |

Let's inspect the actual model structure:

In [ ]:
# Print high-level model structure
print("=" * 60)
print("TOP-LEVEL MODULES")
print("=" * 60)
for name, module in model.named_children():
    param_count = sum(p.numel() for p in module.parameters()) / 1e6
    print(f"  {name:20s}  {param_count:8.1f}M params")

total = sum(p.numel() for p in model.parameters()) / 1e6
print(f"  {'TOTAL':20s}  {total:8.1f}M params")

In [ ]:
# Inspect the vision encoder structure
print("=" * 60)
print("VISION ENCODER")
print("=" * 60)
for name, module in model.model.visual.named_children():
    param_count = sum(p.numel() for p in module.parameters()) / 1e6
    print(f"  {name:30s}  {param_count:8.2f}M params  {type(module).__name__}")

In [ ]:
# Inspect one decoder layer of the language model
print("=" * 60)
print("ONE DECODER LAYER (layer 0)")
print("=" * 60)
layer0 = model.model.layers[0]
for name, module in layer0.named_children():
    param_count = sum(p.numel() for p in module.parameters()) / 1e6
    print(f"  {name:30s}  {param_count:8.2f}M params")

# Show attention head configuration
attn = layer0.self_attn
print(f"\n  Q proj weight shape: {attn.q_proj.weight.shape}  (num_heads={model.config.text_config.num_attention_heads}, head_dim={model.config.text_config.head_dim})")
print(f"  K proj weight shape: {attn.k_proj.weight.shape}  (num_kv_heads={model.config.text_config.num_key_value_heads})")
print(f"  V proj weight shape: {attn.v_proj.weight.shape}")
print(f"  O proj weight shape: {attn.o_proj.weight.shape}")

---
# Part II -- The Vision Encoder: From Pixels to Patch Embeddings

The vision encoder converts a raw image into a sequence of **visual tokens** that the language model can process.

## Step 1: Patch Embedding (3D Convolution)

The image is divided into non-overlapping **16x16 pixel patches** using a 3D convolution. The 3D conv also handles the temporal dimension (for video: `temporal_patch_size=2`).

```
Image: 448 x 448 x 3 (RGB)
  |  3D Conv: kernel=(2, 16, 16), stride=(2, 16, 16)
  v
Patches: 28 x 28 = 784 patches, each 1024-dim
  (448/16 = 28 patches per side)
```

For a single image (T=1), the temporal dimension is effectively 1, so this reduces to a 2D operation.

In [ ]:
# Load a sample image
IMAGE_URL = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
response = requests.get(IMAGE_URL)
image = Image.open(BytesIO(response.content)).convert("RGB")

print(f"Original image size: {image.size} (W x H)")
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title(f"Input Image ({image.size[0]}x{image.size[1]})")
plt.axis("off")
plt.show()

In [ ]:
# Prepare the message in Qwen3-VL chat format
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": IMAGE_URL},
            {"type": "text", "text": "Describe this image in detail."},
        ],
    }
]

# The processor applies the chat template, tokenizes text, AND preprocesses images
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

# Move to GPU
inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

print("\n=== Processor Outputs ===")
for k, v in inputs.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:25s}  shape={str(list(v.shape)):20s}  dtype={v.dtype}")
    elif isinstance(v, list):
        print(f"  {k:25s}  list of length {len(v)}")
    else:
        print(f"  {k:25s}  {type(v).__name__}")

In [ ]:
# Examine the pixel values (preprocessed image)
pixel_values = inputs["pixel_values"]
print(f"pixel_values shape: {pixel_values.shape}")
print(f"  -> This represents the image broken into temporal-spatial chunks")
print(f"  -> The vision encoder's 3D conv will process this into patches")

# Examine the image grid (how the image was resized/tiled)
if "image_grid_thw" in inputs:
    grid = inputs["image_grid_thw"]
    print(f"\nimage_grid_thw: {grid}")
    print(f"  -> T={grid[0,0].item()}, H_patches={grid[0,1].item()}, W_patches={grid[0,2].item()}")
    print(f"  -> Total vision patches before merge: {grid[0,0].item() * grid[0,1].item() * grid[0,2].item()}")
    print(f"  -> After 2x2 spatial merge: {grid[0,0].item() * (grid[0,1].item()//2) * (grid[0,2].item()//2)} visual tokens")

In [ ]:
# Look at the input token sequence -- where are the image placeholders?
input_ids = inputs["input_ids"][0]
tokenizer = processor.tokenizer

# Special token IDs from config
VISION_START = 151652
VISION_END = 151653
IMAGE_TOKEN = 151655

num_image_tokens = (input_ids == IMAGE_TOKEN).sum().item()
total_tokens = len(input_ids)

print(f"Total input tokens: {total_tokens}")
print(f"Image placeholder tokens: {num_image_tokens}")
print(f"Text tokens: {total_tokens - num_image_tokens}")

# Decode the first and last few tokens to see the structure
print(f"\n--- First 20 tokens ---")
for i in range(min(20, len(input_ids))):
    tid = input_ids[i].item()
    if tid == IMAGE_TOKEN:
        label = "<|image_pad|>"
    elif tid == VISION_START:
        label = "<|vision_start|>"
    elif tid == VISION_END:
        label = "<|vision_end|>"
    else:
        label = tokenizer.decode([tid])
    print(f"  [{i:4d}] {tid:8d}  {label}")

print(f"  ... ({num_image_tokens} image tokens in the middle) ...")

print(f"\n--- Last 10 tokens ---")
for i in range(max(0, len(input_ids)-10), len(input_ids)):
    tid = input_ids[i].item()
    if tid == IMAGE_TOKEN:
        label = "<|image_pad|>"
    elif tid == VISION_START:
        label = "<|vision_start|>"
    elif tid == VISION_END:
        label = "<|vision_end|>"
    else:
        label = tokenizer.decode([tid])
    print(f"  [{i:4d}] {tid:8d}  {label}")

### What just happened?

The processor created a token sequence like:
```
<|im_start|> system ... <|im_end|>
<|im_start|> user
<|vision_start|> <|image_pad|> <|image_pad|> ... <|image_pad|> <|vision_end|>
Describe this image in detail.
<|im_end|>
<|im_start|> assistant
```

The `<|image_pad|>` tokens are **placeholders** -- they will be replaced by actual visual embeddings from the vision encoder during the forward pass.

## Step 2: Inside the Vision Encoder

Let's manually trace the vision encoder to see each transformation.

In [ ]:
# Access the vision encoder
visual = model.model.visual

print("Vision Encoder Components:")
print(f"  Patch embed: {type(visual.patch_embed).__name__}")
print(f"    Conv3D weight shape: {visual.patch_embed.proj.weight.shape}")
print(f"    -> kernel: (temporal={2}, H={16}, W={16}), in_channels=3, out_channels=1024")
print(f"  Transformer blocks: {len(visual.blocks)} layers")
print(f"  Merger (final): projects 1024 -> 2048 with 2x2 spatial merge")

# Show DeepStack merger layers
if hasattr(visual, 'deepstack_mergers'):
    print(f"  DeepStack mergers: {len(visual.deepstack_mergers)} layers")
    print(f"    Inject visual features at ViT layers: {model.config.vision_config.deepstack_visual_indexes}")

In [ ]:
# Run JUST the patch embedding to see what comes out
with torch.no_grad():
    patch_embeds = visual.patch_embed(pixel_values)

print(f"After patch embedding:")
print(f"  Input:  pixel_values   {list(pixel_values.shape)}")
print(f"  Output: patch_embeds   {list(patch_embeds.shape)}")
print(f"  -> {patch_embeds.shape[0]} patches, each {patch_embeds.shape[1]}-dimensional")
print(f"  -> Each patch represents a 16x16 pixel region of the image")

In [ ]:
# Visualize what "patches" look like
# Resize image to what the model actually sees
if "image_grid_thw" in inputs:
    grid = inputs["image_grid_thw"][0]
    T, H_patches, W_patches = grid[0].item(), grid[1].item(), grid[2].item()
    patch_size = 16
    
    # Show the patch grid overlaid on the image
    resized = image.resize((W_patches * patch_size, H_patches * patch_size))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Left: original image
    axes[0].imshow(image)
    axes[0].set_title(f"Original ({image.size[0]}x{image.size[1]})")
    axes[0].axis("off")
    
    # Right: resized with patch grid
    axes[1].imshow(resized)
    # Draw patch grid
    for i in range(0, W_patches * patch_size + 1, patch_size):
        axes[1].axvline(x=i, color='red', linewidth=0.5, alpha=0.5)
    for j in range(0, H_patches * patch_size + 1, patch_size):
        axes[1].axhline(y=j, color='red', linewidth=0.5, alpha=0.5)
    axes[1].set_title(f"Resized ({W_patches*patch_size}x{H_patches*patch_size}) with {H_patches}x{W_patches}={H_patches*W_patches} patches")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nAfter 2x2 spatial merge: {H_patches//2} x {W_patches//2} = {(H_patches//2)*(W_patches//2)} visual tokens")
    print(f"Each visual token represents a {patch_size*2}x{patch_size*2} = {(patch_size*2)**2} pixel region")

## Step 3: Spatial Merge

After the ViT processes all patches, a **spatial merge** operation groups every 2x2 block of adjacent patches into a single visual token. This reduces the sequence length by 4x:

```
Before merge (28x28 = 784 patches, each 1024-dim):
 [p00] [p01] [p02] [p03] ...
 [p10] [p11] [p12] [p13] ...
 [p20] [p21] [p22] [p23] ...
 [p30] [p31] [p32] [p33] ...

After 2x2 merge (14x14 = 196 tokens, each 2048-dim):
 [concat(p00,p01,p10,p11)] [concat(p02,p03,p12,p13)] ...
 [concat(p20,p21,p30,p31)] [concat(p22,p23,p32,p33)] ...

The merger MLP: concat(4 * 1024) = 4096 -> MLP -> 2048
(projects to match the LLM hidden_size)
```

In [ ]:
# Inspect the merger MLP
merger = visual.merger
print("Spatial Merger Architecture:")
for name, param in merger.named_parameters():
    print(f"  {name:30s}  {str(list(param.shape)):20s}  ({param.numel()/1e3:.1f}K params)")
print(f"\n  Flow: [{4 * 1024}] --MLP--> [{model.config.text_config.hidden_size}]")
print(f"  (4 patches of 1024-dim concatenated, then projected to 2048)")

---
# Part III -- Vision-Language Fusion

Qwen3-VL introduces three key innovations for fusing vision and language:

## 3.1 DeepStack: Multi-Level Visual Feature Injection

Previous VLMs inject visual features only at the **input** of the language model. Qwen3-VL's **DeepStack** extracts features from intermediate layers of the ViT (layers 5, 11, 17) and injects them into corresponding LLM layers:

```
Vision Encoder (24 layers)          Language Model (28 layers)
========================          ========================
 Layer 0-4                         Layer 0-4
 Layer 5  ----merge----->          Layer 5: += deepstack[0]
 Layer 6-10                        Layer 6-10
 Layer 11 ----merge----->          Layer 11: += deepstack[1]
 Layer 12-16                       Layer 12-16
 Layer 17 ----merge----->          Layer 17: += deepstack[2]
 Layer 18-23                       Layer 18-27
 Final    ----merge----->          Input embedding (scatter)
```

**Why DeepStack?** Early ViT layers capture low-level features (edges, textures), middle layers capture parts and patterns, and later layers capture high-level semantics. By injecting at multiple depths, the LLM gets access to fine-grained visual details that would be lost if only the final ViT output were used.

In [ ]:
# Show the DeepStack injection points
ds_indexes = model.config.vision_config.deepstack_visual_indexes
print(f"DeepStack visual extraction layers: {ds_indexes}")
print(f"Total ViT layers: {model.config.vision_config.depth}")
print()

# Each DeepStack merger has its own MLP to project features
if hasattr(visual, 'deepstack_mergers'):
    for i, (idx, m) in enumerate(zip(ds_indexes, visual.deepstack_mergers)):
        param_count = sum(p.numel() for p in m.parameters()) / 1e3
        print(f"  DeepStack merger {i} (from ViT layer {idx}): {param_count:.1f}K params")
        print(f"    -> Extracts features at ViT layer {idx}, merges 2x2, projects to 2048")
        print(f"    -> Injected additively into LLM layer {idx}")
        print()

## 3.2 M-RoPE: Multimodal Rotary Position Embeddings

Standard RoPE encodes a single position dimension (sequence position). **M-RoPE** extends this to **three dimensions**: temporal, height, and width.

The RoPE frequency dimensions are split into three sections:
- **Temporal (T):** 24 dimensions -- encodes frame position (for video) or constant for images
- **Height (H):** 20 dimensions -- encodes vertical position of the patch in the image
- **Width (W):** 20 dimensions -- encodes horizontal position of the patch in the image

Total: 24 + 20 + 20 = 64 pairs = 128 dimensions (= `head_dim`)

```
For text tokens:     position_ids = [t, t, t]  (same position for all 3 dimensions)
For image patches:   position_ids = [t, h, w]  (3D grid position)

RoPE dimension layout (interleaved):
  [T, H, W, T, H, W, T, H, W, ...]   (mrope_interleaved=True)

This means adjacent dimensions encode different spatial axes,
giving the model fine-grained spatial awareness.
```

In [ ]:
# Show M-RoPE configuration
rope_cfg = model.config.text_config.rope_scaling
print(f"M-RoPE configuration:")
print(f"  rope_type: {rope_cfg.get('rope_type', 'default')}")
print(f"  mrope_section: {rope_cfg['mrope_section']}")
print(f"  mrope_interleaved: {rope_cfg.get('mrope_interleaved', False)}")
print(f"  rope_theta: {model.config.text_config.rope_theta:,.0f}")
print(f"  head_dim: {model.config.text_config.head_dim}")
print()
sections = rope_cfg['mrope_section']
print(f"  Dimension allocation:")
print(f"    Temporal: {sections[0]} pairs = {sections[0]*2} dims  (frame/sequence position)")
print(f"    Height:   {sections[1]} pairs = {sections[1]*2} dims  (vertical patch position)")
print(f"    Width:    {sections[2]} pairs = {sections[2]*2} dims  (horizontal patch position)")
print(f"    Total:    {sum(sections)} pairs = {sum(sections)*2} dims = head_dim={model.config.text_config.head_dim}")

## 3.3 The Scatter Operation: Replacing Placeholders with Visual Embeddings

After the vision encoder produces visual tokens, they need to be inserted into the text token sequence. This is done with a **masked scatter**:

```python
# In the model's forward pass (simplified):
inputs_embeds = embed_tokens(input_ids)        # text embedding for ALL tokens
image_mask = (input_ids == image_token_id)      # find placeholder positions
inputs_embeds[image_mask] = image_embeds         # replace with visual embeddings
```

The placeholder `<|image_pad|>` tokens are overwritten with the actual visual features. The model then processes this mixed sequence of text + visual tokens through the decoder.

In [ ]:
# Demonstrate the scatter concept
input_ids = inputs["input_ids"][0]
image_mask = (input_ids == IMAGE_TOKEN)

print(f"Input sequence length: {len(input_ids)}")
print(f"Image placeholder positions: {image_mask.nonzero().squeeze()[:5].tolist()} ... (total: {image_mask.sum().item()})")
print(f"Text token positions: {(~image_mask).sum().item()}")
print()
print("Token type map (first 80):")
type_map = ["V" if image_mask[i] else "T" for i in range(min(80, len(input_ids)))]
print(" ".join(type_map))
print("T=text, V=visual (will be replaced by vision encoder output)")

---
# Part IV -- The Language Backbone

The language model is essentially a **Qwen3** decoder (the same family as the text-only Qwen3-0.6B in nano-vllm), but with two additions:

1. **M-RoPE** instead of standard 1D RoPE
2. **DeepStack injection** at layers 5, 11, and 17

### Decoder Layer Structure (identical to nano-vllm's Qwen3)

```
For each of the 28 layers:
  hidden_states = RMSNorm(hidden_states + residual)      # pre-norm
  hidden_states = GQA_Attention(hidden_states)            # 16Q/8KV heads, head_dim=128
  [if layer in {5, 11, 17}: hidden_states += deepstack]   # DeepStack injection
  hidden_states = RMSNorm(hidden_states + residual)       # pre-norm
  hidden_states = SwiGLU_FFN(hidden_states)               # 2048 -> 6144 -> 2048
```

### Comparison: Qwen3-VL-2B vs Qwen3-0.6B (nano-vllm)

| Feature | Qwen3-0.6B (text) | Qwen3-VL-2B |
|---------|-------------------|-------------------|
| Layers | 28 | 28 |
| Hidden size | 1024 | 2048 |
| Q heads | 16 | 16 |
| KV heads | 8 | 8 |
| Head dim | 128 | 128 |
| FFN intermediate | 3072 | 6144 |
| RoPE type | Standard 1D | M-RoPE 3D |
| RoPE theta | 1,000,000 | 5,000,000 |
| Vision encoder | None | 24-layer ViT |
| DeepStack | No | Yes (layers 5,11,17) |
| Max context | 40,960 | 262,144 |

In [ ]:
# Compare a text-only Qwen3 layer vs Qwen3-VL layer
layer = model.model.layers[0]
print("=" * 60)
print("Decoder Layer 0 -- Weight Shapes")
print("=" * 60)

# Attention
attn = layer.self_attn
print(f"\n[Attention]")
print(f"  Q projection: {list(attn.q_proj.weight.shape)}  (16 heads x 128 dim = 2048)")
print(f"  K projection: {list(attn.k_proj.weight.shape)}  (8 KV heads x 128 dim = 1024)")
print(f"  V projection: {list(attn.v_proj.weight.shape)}")
print(f"  O projection: {list(attn.o_proj.weight.shape)}  (16 heads x 128 -> 2048)")

# Q/K norms (Qwen3 innovation)
if hasattr(attn, 'q_norm'):
    print(f"  Q norm weight: {list(attn.q_norm.weight.shape)}  (per-head RMSNorm)")
    print(f"  K norm weight: {list(attn.k_norm.weight.shape)}")

# MLP
mlp = layer.mlp
print(f"\n[SwiGLU MLP]")
print(f"  Gate proj: {list(mlp.gate_proj.weight.shape)}  (2048 -> 6144)")
print(f"  Up proj:   {list(mlp.up_proj.weight.shape)}  (2048 -> 6144)")
print(f"  Down proj: {list(mlp.down_proj.weight.shape)}  (6144 -> 2048)")

# Layer norms
print(f"\n[RMSNorm]")
print(f"  Input layernorm:  {list(layer.input_layernorm.weight.shape)}")
print(f"  Post-attn norm:   {list(layer.post_attention_layernorm.weight.shape)}")

# Total params per layer
layer_params = sum(p.numel() for p in layer.parameters()) / 1e6
print(f"\n  Total per layer: {layer_params:.1f}M params")
print(f"  x 28 layers = {layer_params * 28:.1f}M params (language backbone)")

---
# Part V -- End-to-End Inference

Now let's run the full inference pipeline and examine what happens at each stage.

In [ ]:
# Run full generation
print("Running inference...")
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True,
    )

# Trim the prompt tokens to get only the generated output
prompt_len = inputs["input_ids"].shape[1]
output_ids = generated_ids[0, prompt_len:]
output_text = processor.tokenizer.decode(output_ids, skip_special_tokens=True)

print(f"\nPrompt tokens:    {prompt_len}")
print(f"Generated tokens: {len(output_ids)}")
print(f"\n{'='*60}")
print(f"MODEL OUTPUT:")
print(f"{'='*60}")
print(output_text)

## 5.1 Profiling: Where Does Time Go?

Let's measure how long each phase of inference takes.

In [ ]:
import time

# Prepare inputs again
inputs2 = processor.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt",
)
inputs2 = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs2.items()}

# Measure prefill (first token)
torch.cuda.synchronize()
t0 = time.perf_counter()
with torch.no_grad():
    out_prefill = model.generate(**inputs2, max_new_tokens=1)
torch.cuda.synchronize()
prefill_time = time.perf_counter() - t0

# Measure full generation (prefill + decode)
inputs3 = processor.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt",
)
inputs3 = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs3.items()}

NUM_TOKENS = 128
torch.cuda.synchronize()
t0 = time.perf_counter()
with torch.no_grad():
    out_full = model.generate(**inputs3, max_new_tokens=NUM_TOKENS)
torch.cuda.synchronize()
full_time = time.perf_counter() - t0

decode_time = full_time - prefill_time
actual_generated = out_full.shape[1] - inputs3["input_ids"].shape[1]

print(f"Prefill ({inputs3['input_ids'].shape[1]} tokens):")
print(f"  Time: {prefill_time*1000:.1f} ms")
print(f"  Throughput: {inputs3['input_ids'].shape[1]/prefill_time:.0f} tok/s")
print(f"\nDecode ({actual_generated} tokens):")
print(f"  Time: {decode_time*1000:.1f} ms")
print(f"  Throughput: {actual_generated/decode_time:.1f} tok/s")
print(f"\nTotal: {full_time*1000:.1f} ms")
print(f"\nNote: Prefill processes ALL input tokens in parallel (compute-bound).")
print(f"      Decode generates ONE token at a time (memory-bound, bottleneck is KV cache reads).")

## 5.2 GPU Memory Breakdown

In [ ]:
# Memory analysis
torch.cuda.empty_cache()

total_vram = torch.cuda.get_device_properties(0).total_mem / 1e9
allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9

# Calculate model parameter memory
model_params_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
model_params_gb = model_params_bytes / 1e9

# Vision encoder params
vision_params_bytes = sum(p.numel() * p.element_size() for p in model.model.visual.parameters())
vision_params_gb = vision_params_bytes / 1e9

# Language model params
lm_params_bytes = model_params_bytes - vision_params_bytes
lm_params_gb = lm_params_bytes / 1e9

print(f"GPU Memory Usage:")
print(f"  Total VRAM:           {total_vram:.1f} GB")
print(f"  Currently allocated:  {allocated:.2f} GB")
print(f"  Currently reserved:   {reserved:.2f} GB")
print(f"\nModel Parameters (bf16):")
print(f"  Vision encoder:       {vision_params_gb:.2f} GB")
print(f"  Language model + head: {lm_params_gb:.2f} GB")
print(f"  Total:                {model_params_gb:.2f} GB")
print(f"\nKV Cache (at 1024 context length, bf16):")
num_layers = model.config.text_config.num_hidden_layers
num_kv_heads = model.config.text_config.num_key_value_heads
head_dim = model.config.text_config.head_dim
ctx_len = 1024
kv_bytes = 2 * num_layers * ctx_len * num_kv_heads * head_dim * 2  # 2 bytes for bf16
print(f"  Per sequence:         {kv_bytes/1e6:.1f} MB")
print(f"  Formula: 2(K+V) x {num_layers} layers x {ctx_len} tokens x {num_kv_heads} KV heads x {head_dim} dim x 2 bytes")

## 5.3 Multi-Image and Multi-Turn Conversations

In [ ]:
# Multi-turn conversation with the same image
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": IMAGE_URL},
            {"type": "text", "text": "What is in this image?"},
        ],
    },
]

# First turn
inputs_t1 = processor.apply_chat_template(
    conversation, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt",
)
inputs_t1 = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs_t1.items()}

with torch.no_grad():
    out_t1 = model.generate(**inputs_t1, max_new_tokens=128, temperature=0.7, do_sample=True)

response_t1 = processor.tokenizer.decode(
    out_t1[0, inputs_t1["input_ids"].shape[1]:], skip_special_tokens=True
)
print(f"User: What is in this image?")
print(f"Assistant: {response_t1}")

# Second turn (text-only follow-up)
conversation.append({"role": "assistant", "content": response_t1})
conversation.append({
    "role": "user",
    "content": [{"type": "text", "text": "How many people are there? What are they doing?"}],
})

inputs_t2 = processor.apply_chat_template(
    conversation, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt",
)
inputs_t2 = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs_t2.items()}

with torch.no_grad():
    out_t2 = model.generate(**inputs_t2, max_new_tokens=128, temperature=0.7, do_sample=True)

response_t2 = processor.tokenizer.decode(
    out_t2[0, inputs_t2["input_ids"].shape[1]:], skip_special_tokens=True
)
print(f"\nUser: How many people are there? What are they doing?")
print(f"Assistant: {response_t2}")
print(f"\n(Turn 1 tokens: {inputs_t1['input_ids'].shape[1]}, Turn 2 tokens: {inputs_t2['input_ids'].shape[1]})")

---
# Part VI -- Comparing VLM vs Text-Only Inference

The fundamental difference: a VLM adds a vision encoder **before** the same decoder-only LLM backbone.

## Architecture Comparison

```
TEXT-ONLY (Qwen3-0.6B via nano-vllm)        VLM (Qwen3-VL-2B)
================================        ================================

    "Hello world"                           Image + "Describe this"
         |                                       |           |
    Tokenizer                              Processor     Image Preprocessor
         |                                       |           |
    Embedding [151936, 1024]                Embedding   Vision Encoder (24L)
         |                                       |           |
         |                                  Scatter: replace placeholders
         |                                       |           |
    28x Decoder Layer                       28x Decoder Layer
    [1024, GQA 16/8]                        [2048, GQA 16/8]
    [1D RoPE]                               [3D M-RoPE]
    [SwiGLU 3072]                           [SwiGLU 6144]
         |                                  [+ DeepStack at L5,11,17]
         |                                       |
    RMSNorm                                 RMSNorm
         |                                       |
    LM Head -> logits                       LM Head -> logits
         |                                       |
    Sample next token                       Sample next token
```

## What's the Same
- Autoregressive decoding (one token at a time)
- KV cache for efficient decode
- GQA (16 Q heads, 8 KV heads)
- SwiGLU FFN with SiLU activation
- RMSNorm (pre-norm architecture)
- Same vocabulary (151,936 tokens)
- Same sampling strategies (temperature, top-k, top-p)

## What's Different
- VLM adds a vision encoder (~400M params)
- M-RoPE (3D) instead of standard RoPE (1D)
- DeepStack injects visual features at multiple depths
- Input sequence contains a mix of text and visual tokens
- Prefill is more expensive (must also run vision encoder)
- KV cache holds entries for both text AND visual tokens

In [ ]:
# Side-by-side parameter count comparison
print(f"{'Component':<30} {'Qwen3-0.6B (text)':>20} {'Qwen3-VL-2B':>20}")
print("=" * 72)

# Qwen3-0.6B numbers (from nano-vllm config)
q3_hidden = 1024
q3_inter = 3072
q3_layers = 28
q3_vocab = 151936
q3_heads = 16
q3_kv = 8
q3_hdim = 128

# Qwen3-VL-2B numbers
vl_hidden = 2048
vl_inter = 6144
vl_layers = 28

# Embedding
q3_embed = q3_vocab * q3_hidden / 1e6
vl_embed = q3_vocab * vl_hidden / 1e6
print(f"{'Embedding (tied):':<30} {q3_embed:>18.1f}M {vl_embed:>18.1f}M")

# Attention per layer
q3_attn = (q3_hidden * q3_heads * q3_hdim + 2 * q3_hidden * q3_kv * q3_hdim + q3_heads * q3_hdim * q3_hidden) / 1e6
vl_attn = (vl_hidden * q3_heads * q3_hdim + 2 * vl_hidden * q3_kv * q3_hdim + q3_heads * q3_hdim * vl_hidden) / 1e6
print(f"{'Attention (per layer):':<30} {q3_attn:>18.1f}M {vl_attn:>18.1f}M")

# FFN per layer
q3_ffn = (2 * q3_hidden * q3_inter + q3_inter * q3_hidden) / 1e6
vl_ffn = (2 * vl_hidden * vl_inter + vl_inter * vl_hidden) / 1e6
print(f"{'FFN (per layer):':<30} {q3_ffn:>18.1f}M {vl_ffn:>18.1f}M")

# Total LLM backbone
q3_backbone = q3_embed + q3_layers * (q3_attn + q3_ffn)
vl_backbone = vl_embed + vl_layers * (vl_attn + vl_ffn)
print(f"{'LLM backbone total:':<30} {q3_backbone:>18.1f}M {vl_backbone:>18.1f}M")

# Vision encoder
vision_m = sum(p.numel() for p in model.model.visual.parameters()) / 1e6
print(f"{'Vision encoder:':<30} {'N/A':>20} {vision_m:>18.1f}M")

print("=" * 72)
print(f"{'TOTAL:':<30} {q3_backbone:>18.1f}M {vl_backbone + vision_m:>18.1f}M")

## Key Insight: Visual Token Cost

The number of visual tokens directly impacts inference speed and memory:

In [ ]:
# Calculate visual token counts for different image sizes
patch_size = 16
spatial_merge = 2

print(f"{'Image Size':>12} {'Patches':>10} {'After Merge':>12} {'% of 4K context':>16}")
print("-" * 55)
for h, w in [(224, 224), (448, 448), (672, 672), (896, 896), (1344, 1344)]:
    h_patches = h // patch_size
    w_patches = w // patch_size
    total_patches = h_patches * w_patches
    after_merge = (h_patches // spatial_merge) * (w_patches // spatial_merge)
    pct = after_merge / 4096 * 100
    print(f"{h}x{w:>4}   {total_patches:>8}   {after_merge:>10}   {pct:>13.1f}%")

print(f"\nEach visual token occupies the same KV cache space as a text token.")
print(f"A 672x672 image uses ~441 tokens = equivalent to ~330 words of text.")
print(f"Higher resolution = more visual detail but more memory and compute.")

---
## Summary: The Complete VLM Inference Pipeline

```
Step 1: IMAGE PREPROCESSING (CPU)
  - Resize image to fit model constraints
  - Normalize pixel values
  - Compute image grid (T, H_patches, W_patches)

Step 2: TEXT TOKENIZATION (CPU)
  - Apply chat template (system + user + assistant)
  - Insert <|image_pad|> placeholder tokens
  - Compute 3D position IDs (T, H, W) via M-RoPE

Step 3: VISION ENCODING (GPU, parallel)
  - 3D Conv patchify: pixels -> 1024-dim patch embeddings
  - 24-layer ViT with 2D RoPE
  - Extract DeepStack features at layers 5, 11, 17
  - 2x2 spatial merge: patches -> 2048-dim visual tokens

Step 4: SCATTER (GPU)
  - Embed all input tokens (text + placeholders)
  - Replace <|image_pad|> positions with visual embeddings

Step 5: PREFILL (GPU, compute-bound)
  - Run all tokens through 28 decoder layers
  - DeepStack injection at layers 5, 11, 17
  - M-RoPE: text gets 1D positions, image tokens get 3D positions
  - Fill the KV cache
  - Compute logits for the LAST token only

Step 6: DECODE LOOP (GPU, memory-bound)
  - For each new token:
    - Run ONLY the new token through 28 layers
    - Attend to full KV cache (text + visual tokens)
    - Sample next token from logits
    - Append to KV cache
  - Until EOS or max_tokens

Step 7: DETOKENIZE (CPU)
  - Convert output token IDs back to text
```

### What nano-vllm Optimizes (and Would Need for VLM)

All the optimizations from nano-vllm (see `TUTORIAL.md`) apply to Steps 5 and 6:

| Optimization | VLM Applicability |
|---|---|
| **Paged Attention** | Yes -- visual tokens also stored in paged KV cache |
| **Continuous Batching** | Yes -- schedule vision+text prefill, then decode |
| **CUDA Graphs** | Yes -- decode step is identical (single token) |
| **Flash Attention** | Yes -- same O(N) memory attention |
| **torch.compile** | Yes -- RMSNorm, activation, sampling |
| **Prefix Caching** | Yes -- shared system prompts cached; visual tokens could be cached if same image |
| **Tensor Parallelism** | Yes -- shard both vision encoder and LLM |

The main addition for VLM serving would be:
1. A vision encoder forward pass before prefill
2. 3D position ID computation (M-RoPE)
3. DeepStack feature management
4. Image preprocessing pipeline

---
## Bonus: Inspect Attention Patterns

Let's see how the model attends to visual vs text tokens.

In [ ]:
# Run a forward pass with output_attentions=True (expensive, just 1 token decode)
# We'll use a short prompt to keep memory manageable
short_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": IMAGE_URL},
            {"type": "text", "text": "What color is the sky?"},
        ],
    }
]

short_inputs = processor.apply_chat_template(
    short_messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt",
)
short_inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in short_inputs.items()}

# Count visual vs text tokens
ids = short_inputs["input_ids"][0]
is_visual = (ids == IMAGE_TOKEN)
n_visual = is_visual.sum().item()
n_text = len(ids) - n_visual

print(f"Sequence: {len(ids)} tokens ({n_visual} visual + {n_text} text)")
print(f"Visual tokens are positions {is_visual.nonzero()[0].item()} to {is_visual.nonzero()[-1].item()}")
print(f"\nDuring decode, the model attends to ALL of these tokens.")
print(f"Visual tokens act as a 'memory' that the model can query for image information.")

In [ ]:
# Generate with this short prompt
with torch.no_grad():
    short_out = model.generate(
        **short_inputs,
        max_new_tokens=64,
        temperature=0.7,
        do_sample=True,
    )

response = processor.tokenizer.decode(
    short_out[0, short_inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)
print(f"Q: What color is the sky?")
print(f"A: {response}")

---
## Further Exploration

Now that you understand how VLM inference works, here are some directions to explore:

1. **Extend nano-vllm for VLM support**: Add a vision encoder, M-RoPE, and DeepStack to the nano-vllm codebase
2. **Profile with different image resolutions**: Measure how visual token count affects prefill and decode speed
3. **Compare vision encoders**: SigLIP (used in LLaVA) vs ViT (used in Qwen-VL) -- different patchification strategies
4. **Study prefix caching for VLMs**: If the same image appears in multiple requests, its KV cache could be shared
5. **Video understanding**: Qwen3-VL handles video with `temporal_patch_size=2` -- explore how temporal tokens work
6. **Quantization**: Try loading with `load_in_4bit=True` to see how much memory can be saved

### Reading Order for the Codebase

If you want to read the HuggingFace transformers source for Qwen3-VL:

1. `transformers/models/qwen3_vl/configuration_qwen3_vl.py` -- config structure
2. `transformers/models/qwen3_vl/modeling_qwen3_vl.py` -- full model:
   - `Qwen3VLVisionPatchEmbed` -- 3D conv patchification
   - `Qwen3VLVisionBlock` -- ViT transformer block
   - `Qwen3VLVisionPatchMerger` -- 2x2 spatial merge
   - `Qwen3VLVisionModel` -- full vision encoder with DeepStack extraction
   - `Qwen3VLTextModel` -- language model with DeepStack injection + M-RoPE
   - `Qwen3VLModel` -- combined model (scatter + forward)
   - `Qwen3VLForConditionalGeneration` -- generation wrapper
3. `transformers/models/qwen3_vl/processing_qwen3_vl.py` -- processor